# MedGemma 27B Multimodal 4-bit Feasibility Test

This notebook performs no training and writes no model artifacts. It tests whether `unsloth/medgemma-27b-it` can be dynamically loaded in 4-bit on the DGX Spark, verifies text and image inference, attaches a language-only LoRA with the vision tower frozen, and repeats inference.

Run the cells in order inside the `unsloth-notebook` container. Stop if the load cell exhausts memory.

In [ ]:
import os
from pathlib import Path

HF_HUB_CACHE = "/root/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = HF_HUB_CACHE
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "garbage_collection_threshold:0.5,max_split_size_mb:256")
os.environ.setdefault("UNSLOTH_ENABLE_FLEX_ATTENTION", "0")

import unsloth
import psutil
import torch
import transformers
from PIL import Image, ImageDraw
from unsloth import FastVisionModel

MODEL_ID = "unsloth/medgemma-27b-it"
MAX_SEQ_LENGTH = 4096
SEED = 3407
LORA_R = 32
LORA_ALPHA = 32
IMAGE_PATH = Path("/tmp/medgemma_feasibility_vision_test.png")

assert torch.cuda.is_available(), "CUDA is required"
print(f"unsloth={unsloth.__version__}")
print(f"torch={torch.__version__}")
print(f"transformers={transformers.__version__}")
print(f"GPU={torch.cuda.get_device_name(0)}")
print(f"model={MODEL_ID}")
print(f"hf_hub_cache={HF_HUB_CACHE}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth=2026.5.7
torch=2.10.0a0+b558c986e8.nv25.11
transformers=5.10.0.dev0
GPU=NVIDIA GB10
model=unsloth/medgemma-27b-it
hf_hub_cache=/root/.cache/huggingface


In [2]:
def gib(value: int | float) -> float:
    return float(value) / (1024 ** 3)


def memory_snapshot(label: str) -> dict[str, float]:
    process = psutil.Process()
    virtual = psutil.virtual_memory()
    snapshot = {
        "process_rss_gib": gib(process.memory_info().rss),
        "system_used_gib": gib(virtual.used),
        "system_available_gib": gib(virtual.available),
        "cuda_allocated_gib": gib(torch.cuda.memory_allocated()),
        "cuda_reserved_gib": gib(torch.cuda.memory_reserved()),
        "cuda_peak_allocated_gib": gib(torch.cuda.max_memory_allocated()),
    }
    print(label)
    for name, value in snapshot.items():
        print(f"  {name}: {value:.2f}")
    return snapshot


def generate(messages, max_new_tokens: int = 160) -> str:
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)
    input_length = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )
    return processor.decode(output[0, input_length:], skip_special_tokens=True).strip()


torch.cuda.reset_peak_memory_stats()
baseline_memory = memory_snapshot("Before model load")

Before model load
  process_rss_gib: 1.32
  system_used_gib: 12.39
  system_available_gib: 109.30
  cuda_allocated_gib: 0.01
  cuda_reserved_gib: 0.02
  cuda_peak_allocated_gib: 0.01


In [3]:
model, processor = FastVisionModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
    device_map={"": 0},
    cache_dir=HF_HUB_CACHE,
)
FastVisionModel.for_inference(model)

loaded_memory = memory_snapshot("After 4-bit model load")
print(f"model_class={type(model).__name__}")
print(f"processor_class={type(processor).__name__}")
print(f"architecture={model.config.architectures}")
print(f"vision_model_type={model.config.vision_config.model_type}")
print(f"text_model_type={model.config.text_config.model_type}")

==((====))==  Unsloth 2026.5.7: Fast Gemma3 patching. Transformers: 5.10.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

After 4-bit model load
  process_rss_gib: 1.10
  system_used_gib: 27.34
  system_available_gib: 94.35
  cuda_allocated_gib: 15.75
  cuda_reserved_gib: 15.87
  cuda_peak_allocated_gib: 15.75
model_class=Gemma3ForConditionalGeneration
processor_class=Gemma3Processor
architecture=['Gemma3ForConditionalGeneration']
vision_model_type=siglip_vision_model
text_model_type=gemma3_text


In [4]:
parameter_bytes = sum(parameter.numel() * parameter.element_size() for parameter in model.parameters())
parameter_dtypes = {}
for parameter in model.parameters():
    key = str(parameter.dtype)
    parameter_dtypes[key] = parameter_dtypes.get(key, 0) + parameter.numel()

quantized_modules = sum(1 for module in model.modules() if "4bit" in type(module).__name__.lower())
vision_parameter_names = [name for name, _ in model.named_parameters() if "vision" in name.lower()]

print(f"parameter_storage_gib={gib(parameter_bytes):.2f}")
print(f"quantized_4bit_modules={quantized_modules}")
print(f"parameter_dtypes={parameter_dtypes}")
print(f"vision_parameter_tensors={len(vision_parameter_names)}")
print("example_vision_parameters:")
for name in vision_parameter_names[:8]:
    print(f"  {name}")

assert quantized_modules > 0, "No 4-bit modules were found"
assert vision_parameter_names, "No vision tower parameters were found"

parameter_storage_gib=15.34
quantized_4bit_modules=434
parameter_dtypes={'torch.bfloat16': 1832562672, 'torch.float32': 1482368, 'torch.uint8': 12799180800}
vision_parameter_tensors=437
example_vision_parameters:
  model.vision_tower.embeddings.patch_embedding.weight
  model.vision_tower.embeddings.patch_embedding.bias
  model.vision_tower.embeddings.position_embedding.weight
  model.vision_tower.encoder.layers.0.layer_norm1.weight
  model.vision_tower.encoder.layers.0.layer_norm1.bias
  model.vision_tower.encoder.layers.0.self_attn.k_proj.weight
  model.vision_tower.encoder.layers.0.self_attn.k_proj.bias
  model.vision_tower.encoder.layers.0.self_attn.v_proj.weight


In [5]:
text_messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful medical research assistant."}],
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": "Hi"}],
    },
]
text_response_before_lora = generate(text_messages, max_new_tokens=80)
assert text_response_before_lora, "Text inference returned an empty response"
print(text_response_before_lora)
memory_snapshot("After base text inference")

Hi there! How can I help you with your medical research today? I'm ready to assist with tasks like:

*   **Literature searching:** Finding relevant articles, studies, and reviews on specific topics.
*   **Summarizing information:** Condensing complex research papers or articles into key points.
*   **Identifying key concepts and terms:** Helping you understand the terminology used in a
After base text inference
  process_rss_gib: 1.91
  system_used_gib: 28.86
  system_available_gib: 92.83
  cuda_allocated_gib: 15.75
  cuda_reserved_gib: 16.18
  cuda_peak_allocated_gib: 16.03


{'process_rss_gib': 1.9080696105957031,
 'system_used_gib': 28.860408782958984,
 'system_available_gib': 92.82893371582031,
 'cuda_allocated_gib': 15.754605770111084,
 'cuda_reserved_gib': 16.181640625,
 'cuda_peak_allocated_gib': 16.02535915374756}

In [ ]:
image = Image.new("RGB", (896, 896), color=(238, 238, 232))
draw = ImageDraw.Draw(image)
draw.rectangle((70, 70, 826, 826), outline=(25, 25, 25), width=10)
draw.ellipse((140, 180, 400, 700), fill=(48, 92, 170), outline=(15, 35, 70), width=8)
draw.ellipse((496, 180, 756, 700), fill=(196, 56, 48), outline=(85, 20, 18), width=8)
draw.rectangle((358, 330, 538, 566), fill=(42, 150, 82), outline=(15, 70, 36), width=8)
draw.text((330, 105), "VISION TEST", fill=(10, 10, 10))
image.save(IMAGE_PATH)
print(f"image={IMAGE_PATH} size={image.size} mode={image.mode}")

image_messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a precise visual assistant."}],
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "This is a synthetic vision test card, not a medical image. Describe its visible colors, shapes, and text. Do not provide medical interpretation."},
        ],
    },
]
image_response_before_lora = generate(image_messages, max_new_tokens=160)
assert image_response_before_lora, "Image inference returned an empty response"
print(image_response_before_lora)
memory_snapshot("After base image inference")

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)
FastVisionModel.for_inference(model)

trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
vision_trainable = [name for name in trainable_names if "vision" in name.lower()]

print(f"trainable_parameters={trainable_parameters:,}")
print(f"trainable_tensors={len(trainable_names)}")
print(f"vision_trainable_tensors={len(vision_trainable)}")
for name in trainable_names[:12]:
    print(f"  {name}")

assert trainable_parameters > 0, "LoRA attachment produced no trainable parameters"
assert not vision_trainable, f"Vision parameters unexpectedly trainable: {vision_trainable[:5]}"
lora_memory = memory_snapshot("After language-only LoRA attachment")

In [ ]:
text_response_after_lora = generate(text_messages, max_new_tokens=80)
image_response_after_lora = generate(image_messages, max_new_tokens=200)

assert text_response_after_lora, "Post-LoRA text inference returned an empty response"
assert image_response_after_lora, "Post-LoRA image inference returned an empty response"

print("TEXT AFTER LORA:")
print(text_response_after_lora)
print("\nIMAGE AFTER LORA:")
print(image_response_after_lora)
final_memory = memory_snapshot("After post-LoRA text and image inference")

In [ ]:
print("FEASIBILITY TEST PASSED")
print(f"Model: {MODEL_ID}")
print(f"4-bit modules: {quantized_modules}")
print(f"Parameter storage: {gib(parameter_bytes):.2f} GiB")
print(f"Peak CUDA allocation: {final_memory['cuda_peak_allocated_gib']:.2f} GiB")
print(f"Final system available memory: {final_memory['system_available_gib']:.2f} GiB")
print(f"Vision trainable tensors: {len(vision_trainable)}")
print("No training or model save was performed.")